In [0]:
master_path = "/Volumes/workspace/default/week7_data/customer_master.csv"
incremental_path = "/Volumes/workspace/default/week7_data/customer_incremental.csv"

master_df = spark.read.option("header", True).option("inferSchema", True).csv(master_path)

incremental_df = spark.read.option("header", True).option("inferSchema", True).csv(incremental_path)

print("Master Dataset")
display(master_df)

print("Incremental Dataset")
display(incremental_df)

Master Dataset


customer_id,name,email,city,status
101,Rahul,rahul@gmail.com,Delhi,Active
102,Priya,priya@gmail.com,Mumbai,Active
103,Aman,null,Jaipur,Active
104,Neha,neha@gmail.com,Pune,Inactive
105,Rohit,rohit@gmail.com,Delhi,Active
105,Rohit,rohit@gmail.com,Delhi,Active


Incremental Dataset


customer_id,name,email,city,status
102,Priya,priya@gmail.com,Bangalore,Active
104,Neha,neha@gmail.com,Pune,Active
106,Arjun,arjun@gmail.com,Chennai,Active
107,Simran,simran@gmail.com,Chandigarh,Active


In [0]:
# Remove null values
master_clean = master_df.dropna()
incremental_clean = incremental_df.dropna()

# Remove duplicate records based on customer_id
master_clean = master_clean.dropDuplicates(["customer_id"])
incremental_clean = incremental_clean.dropDuplicates(["customer_id"])

print("Cleaned Master Dataset")
display(master_clean)

print("Cleaned Incremental Dataset")
display(incremental_clean)

Cleaned Master Dataset


customer_id,name,email,city,status
105,Rohit,rohit@gmail.com,Delhi,Active
104,Neha,neha@gmail.com,Pune,Inactive
102,Priya,priya@gmail.com,Mumbai,Active
101,Rahul,rahul@gmail.com,Delhi,Active


Cleaned Incremental Dataset


customer_id,name,email,city,status
104,Neha,neha@gmail.com,Pune,Active
106,Arjun,arjun@gmail.com,Chennai,Active
107,Simran,simran@gmail.com,Chandigarh,Active
102,Priya,priya@gmail.com,Bangalore,Active


In [0]:
print("Master Row Count:", master_clean.count())
print("Incremental Row Count:", incremental_clean.count())

print("Master Duplicate Count:")
print(master_clean.count() - master_clean.dropDuplicates(["customer_id"]).count())

print("Incremental Duplicate Count:")
print(incremental_clean.count() - incremental_clean.dropDuplicates(["customer_id"]).count())

Master Row Count: 4
Incremental Row Count: 4
Master Duplicate Count:
0
Incremental Duplicate Count:
0


In [0]:
delta_table_name = "workspace.default.customer_delta"

master_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(delta_table_name)

print("Delta table created successfully")

display(spark.table(delta_table_name))

Delta table created successfully


customer_id,name,email,city,status
105,Rohit,rohit@gmail.com,Delhi,Active
104,Neha,neha@gmail.com,Pune,Inactive
102,Priya,priya@gmail.com,Mumbai,Active
101,Rahul,rahul@gmail.com,Delhi,Active


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "workspace.default.customer_delta"
)

delta_table.alias("target") \
    .merge(
        incremental_clean.alias("source"),
        "target.customer_id = source.customer_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("MERGE operation completed successfully")

MERGE operation completed successfully


In [0]:
final_df = spark.table("workspace.default.customer_delta")

print("Final Dataset After MERGE")

display(
    final_df.orderBy("customer_id")
)

Final Dataset After MERGE


customer_id,name,email,city,status
101,Rahul,rahul@gmail.com,Delhi,Active
102,Priya,priya@gmail.com,Bangalore,Active
104,Neha,neha@gmail.com,Pune,Active
105,Rohit,rohit@gmail.com,Delhi,Active
106,Arjun,arjun@gmail.com,Chennai,Active
107,Simran,simran@gmail.com,Chandigarh,Active


In [0]:
print("Final Row Count:", final_df.count())

duplicate_count = (
    final_df.count()
    - final_df.dropDuplicates(["customer_id"]).count()
)

print("Duplicate Customer IDs:", duplicate_count)

print("Updated and Inserted Records")

display(
    final_df
    .filter(final_df.customer_id.isin(102, 104, 106, 107))
    .orderBy("customer_id")
)

Final Row Count: 6
Duplicate Customer IDs: 0
Updated and Inserted Records


customer_id,name,email,city,status
102,Priya,priya@gmail.com,Bangalore,Active
104,Neha,neha@gmail.com,Pune,Active
106,Arjun,arjun@gmail.com,Chennai,Active
107,Simran,simran@gmail.com,Chandigarh,Active


In [0]:
history_df = spark.sql("""
DESCRIBE HISTORY workspace.default.customer_delta
""")

display(history_df)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-06T14:16:02.000Z,74223451951988,studyithimanshu@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2171576820903134),6543e9e7-b2e5-4495-9d49-bb5d5ef5b313,0706-140520-rrqdb09-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3756, p25FileSize -> 1925, numDeletionVectorsRemoved -> 1, minFileSize -> 1925, numAddedFiles -> 1, maxFileSize -> 1925, p75FileSize -> 1925, p50FileSize -> 1925, numAddedBytes -> 1925)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
1,2026-07-06T14:15:59.000Z,74223451951988,studyithimanshu@gmail.com,MERGE,"Map(predicate -> [""(customer_id#12016 = customer_id#12039)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2171576820903134),6543e9e7-b2e5-4495-9d49-bb5d5ef5b313,0706-140520-rrqdb09-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1889, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 6979, materializeSourceTimeMs -> 604, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3042, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3185)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-07-06T14:14:11.000Z,74223451951988,studyithimanshu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2171576820903134),3a699b06-dc2b-477f-81ea-12fb1e230b78,0706-140520-rrqdb09-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 4, numOutputBytes -> 1867)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


In [0]:
print("===== DELTA LAKE MERGE ASSIGNMENT SUMMARY =====")
print("Initial Master Records:", master_clean.count())
print("Incremental Records:", incremental_clean.count())
print("Final Records:", final_df.count())
print("Updated Records: 2")
print("Inserted Records: 2")
print("Duplicate Customer IDs: 0")
print("MERGE implementation completed successfully")

===== DELTA LAKE MERGE ASSIGNMENT SUMMARY =====
Initial Master Records: 4
Incremental Records: 4
Final Records: 6
Updated Records: 2
Inserted Records: 2
Duplicate Customer IDs: 0
MERGE implementation completed successfully
